# Stage 03 — Data Transformation
Prototype notebook for the data transformation component.
Run after `2_data_validation.ipynb` passes.

In [11]:
import os
#os.chdir("../")
%pwd

'/home/abood/Desktop/ML_Model_Monitoring_System_for_Credit_Risk'

In [12]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    preprocessor_path: Path

In [13]:
from src.creditrisk.constants import *
from src.creditrisk.utils import read_yaml, create_directories, logger

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])
        return DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            preprocessor_path=config.preprocessor_path
        )

In [14]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from src.creditrisk.utils import logger

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def clean_and_engineer(self, df):
        df_clean = df.copy()
        df_clean = df_clean.drop_duplicates()
        
        # Drop identifier columns early
        df_clean = df_clean.drop(columns=["customer_id", "name"], errors="ignore")

        # Impute missing values before feature engineering
        for col in df_clean.select_dtypes(include=[np.number]).columns:
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        for col in df_clean.select_dtypes(include=["object"]).columns:
            df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

        # Engineered Ratios
        df_clean['loan_to_income_ratio'] = df_clean['credit_limit'] / (df_clean['net_yearly_income'] + 1)
        df_clean['debt_to_income_ratio'] = df_clean['yearly_debt_payments'] / (df_clean['net_yearly_income'] + 1)
        
        # Credit utilization bin
        df_clean['credit_utilization_bin'] = pd.cut(
            df_clean['credit_limit_used(%)'], 
            bins=[-1, 30, 70, 100], 
            labels=['Low', 'Medium', 'High']
        )
        
        # Risk score 
        df_clean['risk_score'] = (df_clean['prev_defaults'] * 50) + (df_clean['default_in_last_6months'] * 100) + (1000 - df_clean['credit_score']) + (df_clean['credit_limit']/1000)
        
        # Drop low-importance demographic features
        cols_to_drop = ["gender", "owns_car", "owns_house", "no_of_children", "migrant_worker", "total_family_members"]
        df_clean = df_clean.drop(columns=[col for col in cols_to_drop if col in df_clean.columns])

        # Outlier removal (skip target)
        numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
        target = "credit_card_default"
        if target in numeric_cols:
            numeric_cols.remove(target)

        outliers_mask = pd.Series([False] * len(df_clean), index=df_clean.index)
        for col in numeric_cols:
            Q1 = df_clean[col].quantile(0.25)
            Q3 = df_clean[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            col_outliers = (df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)
            outliers_mask = outliers_mask | col_outliers.values

        df_clean = df_clean[~outliers_mask]
        logger.info(f"Data cleaned & engineered: {len(df)} rows → {len(df_clean)} rows")
        return df_clean

    def initiate_data_transformation(self):
        # NOTE: Removed `skiprows=1` because our dataset has standard headers
        df = pd.read_csv(self.config.data_path)
        logger.info(f"Loaded data: {df.shape[0]} rows, {df.shape[1]} columns")

        df_clean = self.clean_and_engineer(df)

        # One-hot encode categorical columns
        categorical_cols = df_clean.select_dtypes(include=["object", "category"]).columns.tolist()
        if categorical_cols:
            df_clean = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)
            logger.info(f"One-hot encoded categorical columns. New shape: {df_clean.shape}")

        # Separate features and target
        target_col = "credit_card_default"
        y = df_clean.pop(target_col)
        X = df_clean

        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y, random_state=42, stratify=y, test_size=0.2)
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, random_state=42, stratify=y_temp, test_size=0.25)

        scaler = StandardScaler()
        X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
        X_val   = pd.DataFrame(scaler.transform(X_val),       columns=X.columns)
        X_test  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns)

        logger.info(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

        for name, arr in [("X_train",X_train),("X_val",X_val),("X_test",X_test),
                          ("y_train",y_train),("y_val",y_val),("y_test",y_test)]:
            arr.to_csv(os.path.join(self.config.root_dir, f"{name}.csv"), index=False)

        with open(self.config.preprocessor_path, "wb") as f:
            pickle.dump(scaler, f)
        logger.info("Data transformation complete. Preprocessor and splits saved.")


In [15]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.initiate_data_transformation()
except Exception as e:
    raise e

[2026-05-04 19:49:40,472: INFO: common]: yaml file: config/config.yaml loaded successfully
[2026-05-04 19:49:40,473: INFO: common]: yaml file: params.yaml loaded successfully
[2026-05-04 19:49:40,476: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-05-04 19:49:40,476: INFO: common]: created directory at: artifacts
[2026-05-04 19:49:40,477: INFO: common]: created directory at: artifacts/data_transformation
[2026-05-04 19:49:40,546: INFO: 809150788]: Loaded data: 45528 rows, 19 columns
[2026-05-04 19:49:40,655: INFO: 809150788]: Data cleaned & engineered: 45528 rows → 31224 rows
[2026-05-04 19:49:40,662: INFO: 809150788]: One-hot encoded categorical columns. New shape: (31224, 33)
[2026-05-04 19:49:40,719: INFO: 809150788]: Train: (18734, 32), Val: (6245, 32), Test: (6245, 32)
[2026-05-04 19:49:41,742: INFO: 809150788]: Data transformation complete. Preprocessor and splits saved.


In [16]:
import os
files = os.listdir("artifacts/data_transformation")
for f in sorted(files):
    path = os.path.join("artifacts/data_transformation", f)
    size = os.path.getsize(path)
    print(f"{f:30s}  {size:,} bytes")

X_test.csv                      3,828,951 bytes
X_train.csv                     11,484,557 bytes
X_val.csv                       3,828,975 bytes
scaler.pkl                      2,135 bytes
y_test.csv                      12,510 bytes
y_train.csv                     37,488 bytes
y_val.csv                       12,510 bytes
